# Lab 3: Clustering Standard Errors — Monte Carlo Study

**课程**：经济与商务实证研究方法（RMEB） 2026 Spring  
**主题**：聚类标准误的 Monte Carlo 验证  
**配套讲义**：Week 3 — 面板数据与经典 DiD

## 学习目标

- 在 Stata 中生成一个带序列相关的面板
- 复现 Bertrand-Duflo-Mullainathan (2004) 的 45% 过度拒绝结果
- 对比 iid / robust / 单聚类 / 两维聚类 / Wild Bootstrap 的推断表现
- 改变 AR(1) 系数 ρ 观察推断失真速度

## 参考文献

- Bertrand, M., Duflo, E., & Mullainathan, S. (2004). *QJE* 119(1), 249–275.
- Cameron, A. C., & Miller, D. L. (2015). *JHR* 50(2), 317–372.
- Cameron, A. C., Gelbach, J. B., & Miller, D. L. (2008). *ReStat* 90(3), 414–427.


## 对此Lab的修改
### 1. 时间效应生成
在运行第3部分的代码时提示insufficient observations，说明生成的数据集中没有足够的观测值来进行分析。
将报错报告给copilot，提示第2部分定义的函数之后发现原先在生成时间效应的时候只对第1期生成了时间效应，导致其他期时间效应为空，从而计算出的y也是空的。并给出了修改建议：
```stata
bysort t: gen time_fe = rnormal(0,0.5) if _n == 1
bysort t: replace time_fe = time_fe[1]
```
我对stata代码不熟悉，不了解bysort，要求改为更易读的循环版本，copilot修改为：
```stata
forvalues tt = 1/`t' {
    local draw = rnormal(0, 0.5)
    replace time_fe = `draw' if t == `tt'
}
```
即对每个时期都抽一个随机的时间效应并赋值。

### 2. 生成AR(1)误差项
生成AR(1)误差项时使用了L来表示滞后项，就需要先xtset设置好面板结构。
完成以上修改后，后续运行不再报错。

## 1. 环境准备与包安装

运行前请确保已安装以下 Stata 外部命令（首次运行需要联网）：

In [33]:
clear all
set more off
set seed 20260516

* 安装必要包
/* capture ssc install reghdfe
capture ssc install ftools
capture ssc install boottest
capture ssc install estout */

## 2. 基本 DGP：500 单位 × 10 期面板

AR(1) 序列相关误差，随机选 100 个单位在 t=5 后接受"假政策"（真实效应为 0）。

In [ ]:
program define gen_panel
    syntax, [n(integer 500) t(integer 10) rho(real 0.8) te(real 0.0) seed(integer 0)]
    clear
    set seed `seed'
    // 创建一张空表， n行0列
    set obs `n'
    // 根据行号生成个体的id
    gen unit = _n
    // 生成单位固定效应，从标准正分布中随机抽取
    gen unit_fe = rnormal(0, 1)
    
    * 随机选 1/5 的单位作为处理组
    // 生成一列随机数并按随机数排序
    gen u = runiform()
    sort u
    // 此时位于前 1/5 的单位标记为处理组
    gen treated = (_n <= `n' / 5)
    drop u
    
    // 生成面板数据：每个单位有 t 个时间点
    expand `t'
    // 按unit分组，对于每个个体，生成时间变量 t，表示每个单位的时间顺序
    bysort unit: gen t = _n
    
    // 原来的时间效应只给第一期生成了一个随机数
    /* gen time_fe = rnormal(0, 0.5) if t == 1
    bysort t: egen tfe = mean(time_fe)
    drop time_fe
    rename tfe time_fe */
    
    // 修改后时间效应：应该为每个时期都生成一个时间冲击
    gen time_fe = .
    forvalues tt = 1/`t' {
        local draw = rnormal(0, 0.5)
        replace time_fe = `draw' if t == `tt'
    }
    
    * AR(1) 误差：eps_it = rho * eps_{i,t-1} + u_it
    sort unit t
    gen eps = rnormal(0, 1) if t == 1
    // 因为要使用L.eps，所以必须先设置面板数据结构
    xtset unit t
    replace eps = `rho' * L.eps + rnormal(0, 1) if t > 1 & !missing(L.eps)

    * 处理指示与结果
    gen D = (treated == 1 & t >= 5)
    gen Y = unit_fe + time_fe + `te' * D + eps
end

## 3. 单次运行：观察三种标准误

In [35]:
gen_panel, n(500) t(10) rho(0.8) te(0.0) seed(42)

* iid 标准误
reghdfe Y D, absorb(unit t)
estimates store m_iid

* Robust (HC1)
reghdfe Y D, absorb(unit t) vce(robust)
estimates store m_hc

* 按单位聚类
reghdfe Y D, absorb(unit t) vce(cluster unit)
estimates store m_cl

esttab m_iid m_hc m_cl, star(* 0.10 ** 0.05 *** 0.01) ///
    stats(N r2) mtitles("iid" "HC1" "Cluster") b(3) se(3)


Number of observations (_N) was 0, now 500.
(4,500 observations created)
(5,000 missing values generated)
(500 real changes made)
(500 real changes made)
(500 real changes made)
(500 real changes made)
(500 real changes made)
(500 real changes made)
(500 real changes made)
(500 real changes made)
(500 real changes made)
(500 real changes made)
(4,500 missing values generated)

Panel variable: unit (strongly balanced)
 Time variable: t, 1 to 10
         Delta: 1 unit
(4,500 real changes made)

(MWFE estimator converged in 2 iterations)

HDFE Linear regression                            Number of obs   =      5,000
Absorbing 2 HDFE groups                           F(   1,   4490) =       0.08
                                                  Prob > F        =     0.7770
                                                  R-squared       =     0.6832
                                                  Adj R-squared   =     0.6473
                                                  Within R-sq.

## 4. Monte Carlo：300 次重复的拒绝率对比

这是 BDM(2004) 逻辑的完整复现。真实效应 = 0；若名义 5% 水平下拒绝率高于 5%，则推断失真。

In [36]:
postfile mcresults rep p_iid p_hc p_cl using mc_cluster.dta, replace

local nreps = 300
forvalues r = 1/`nreps' {
    quietly {
        gen_panel, n(500) t(10) rho(0.8) te(0.0) seed(`r')
        
        reghdfe Y D, absorb(unit t)
        local p_iid = 2 * ttail(e(df_r), abs(_b[D] / _se[D]))
        
        reghdfe Y D, absorb(unit t) vce(robust)
        local p_hc = 2 * ttail(e(df_r), abs(_b[D] / _se[D]))
        
        reghdfe Y D, absorb(unit t) vce(cluster unit)
        local p_cl = 2 * ttail(e(df_r), abs(_b[D] / _se[D]))
    }
    post mcresults (`r') (`p_iid') (`p_hc') (`p_cl')
    if mod(`r', 50) == 0 display "完成 `r' / `nreps'"
}
postclose mcresults

use mc_cluster.dta, clear
foreach se in iid hc cl {
    gen reject_`se' = (p_`se' < 0.05)
    quietly summarize reject_`se', meanonly
    display "标准误类型 `se': 拒绝率 = " %5.1f r(mean) * 100 "%"
}




完成 50 / 300
完成 100 / 300
完成 150 / 300
完成 200 / 300
完成 250 / 300
完成 300 / 300



标准误类型 iid: 拒绝率 =  28.7%
标准误类型 hc: 拒绝率 =  29.3%
标准误类型 cl: 拒绝率 =   6.3%


**期望结果**：

- iid 标准误：远高于 5%（可能 30-45%）
- HC1 稳健：仍明显高于 5%
- 按单位聚类：接近 5%

这就是为什么 Cameron-Miller 说：面板数据*必须聚类*。

## 5. 改变 ρ：观察推断退化速度

如果误差没有序列相关（ρ = 0），iid 标准误应当是正确的。随着 ρ 增加，推断失真加剧。

In [37]:
* 跑三组 ρ 的 MC
postfile rho_results rho p_iid p_cl using mc_rho.dta, replace

foreach rho_val in 0.0 0.4 0.8 {
    forvalues r = 1/100 {
        quietly {
            gen_panel, n(500) t(10) rho(`rho_val') te(0.0) seed(`r')
            reghdfe Y D, absorb(unit t)
            local p_iid = 2 * ttail(e(df_r), abs(_b[D] / _se[D]))
            reghdfe Y D, absorb(unit t) vce(cluster unit)
            local p_cl = 2 * ttail(e(df_r), abs(_b[D] / _se[D]))
        }
        post rho_results (`rho_val') (`p_iid') (`p_cl')
    }
}
postclose rho_results

use mc_rho.dta, clear
gen reject_iid = (p_iid < 0.05)
gen reject_cl  = (p_cl  < 0.05)
table rho, stat(mean reject_iid reject_cl)









---------------------------------
        |  reject_iid   reject_cl
--------+------------------------
rho     |                        
  0     |         .04         .03
  .4    |         .14         .03
  .8    |          .2         .05
  Total |    .1266667    .0366667
---------------------------------


## 6. 小聚类数：Wild Bootstrap

当聚类数 G < 50 时，渐近推断不可靠。此时用 Wild Cluster Bootstrap（Cameron-Gelbach-Miller 2008）。

In [38]:
* 只保留前 30 个单位（模拟小 G 情境）
gen_panel, n(30) t(10) rho(0.8) te(0.0) seed(42)

reghdfe Y D, absorb(unit t) vce(cluster unit)

* Wild Cluster Bootstrap（999 次）
boottest D, boottype(wild-t) reps(999) cluster(unit)


Number of observations (_N) was 0, now 30.
(270 observations created)
(300 missing values generated)
(30 real changes made)
(30 real changes made)
(30 real changes made)
(30 real changes made)
(30 real changes made)
(30 real changes made)
(30 real changes made)
(30 real changes made)
(30 real changes made)
(30 real changes made)
(270 missing values generated)

Panel variable: unit (strongly balanced)
 Time variable: t, 1 to 10
         Delta: 1 unit
(270 real changes made)

(MWFE estimator converged in 2 iterations)

HDFE Linear regression                            Number of obs   =        300
Absorbing 2 HDFE groups                           F(   1,     29) =       3.63
Statistics robust to heteroskedasticity           Prob > F        =     0.0668
                                                  R-squared       =     0.7138
                                                  Adj R-squared   =     0.6708
                                                  Within R-sq.    =     0.0261
Nu

## 7. 两维聚类

若误差既沿时间传染（行业波动）又沿空间传染（地区外溢），使用 Cameron-Gelbach-Miller (2011) 两维聚类。

In [39]:
gen_panel, n(500) t(10) rho(0.8) te(0.0) seed(42)

* 假设每 25 个单位构成一个"行业"
gen industry = ceil(unit / 25)

reghdfe Y D, absorb(unit t) vce(cluster unit t)
reghdfe Y D, absorb(unit t) vce(cluster unit industry)


Number of observations (_N) was 0, now 500.
(4,500 observations created)
(5,000 missing values generated)
(500 real changes made)
(500 real changes made)
(500 real changes made)
(500 real changes made)
(500 real changes made)
(500 real changes made)
(500 real changes made)
(500 real changes made)
(500 real changes made)
(500 real changes made)
(4,500 missing values generated)

Panel variable: unit (strongly balanced)
 Time variable: t, 1 to 10
         Delta: 1 unit
(4,500 real changes made)


(MWFE estimator converged in 2 iterations)
> bach & Miller applied.

HDFE Linear regression                            Number of obs   =      5,000
Absorbing 2 HDFE groups                           F(   1,      9) =       0.03
Statistics robust to heteroskedasticity           Prob > F        =     0.8769
                                                  R-squared       =     0.6832
                                                  Adj R-squared   =     0.6472
Number of clusters (unit)    =      

## 8. 练习

1. 把 `rho` 提到 0.95，把 `te` 保持为 0，跑 MC——观察失真程度。
2. 把 `n` 降到 30，用 Wild Bootstrap 与标准聚类做 MC 对比拒绝率差异。
3. 给定 `te = 0.3`（真实有效应），比较不同标准误下*功效*（power）的差异。
4. 用 Python 或 R 复现上述 MC，对比三种语言的速度。

## 9. 延伸阅读

- Abadie, Athey, Imbens, Wooldridge (2023). "When should you adjust standard errors for clustering?" *QJE* 138(1), 1–35.
- MacKinnon & Webb (2020). "Wild Bootstrap Randomization Inference for Few Treated Clusters." *JAE* 35(7), 888–908.